In [771]:
import numpy as np 
import random 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [873]:
df = pd.read_csv(r"C:\Users\rawat\startup_funding.csv")
#df.head(10) seeing first 10 values in data  
#df.shape rows and column are present 
pd.set_option('display.max.rows',3044) # making max rows visible

# Renaming columns for better assessibility 
df.rename(columns={'Date dd/mm/yyyy':'Date','InvestmentnType':'Investment Type',
                   'City  Location':'City Location',
                   'SubVertical':'Sub Vertical'},inplace=True)

# Forward filling city location and date 
df['City Location'] = df['City Location'].ffill()
df['Date'] = df['Date'].ffill()

# Conversion of Amount column in numeric format 
df['Amount in USD'] = pd.to_numeric(df['Amount in USD'],errors = 'coerce')

# Displaying the amount in .2f format 
pd.options.display.float_format = '{:.2f}'.format


## Cleaning columns with ascii kewords 
#'\\xc2\\xa0'  \\xe2\\x80\\x99s  \\xe2\\x80\\x99  \\xc3\\x98r
columns = ['Startup Name','Industry Vertical','Sub Vertical','City Location','Investors Name','Investment Type','Remarks']

#Replaceing all non-ascii/hidden characters with none
#df[columns] = df[columns].replace(r'[^\x00-\x7F]+', None, regex=True)
search = r'\\'  #153
df.isna().sum()
dropping_rows = df[(df['Sub Vertical'].isna())&(df['Amount in USD'].isna())].index
df.drop(dropping_rows,inplace=True)
df.loc[df['Startup Name'].str.contains(search,regex=True),'Startup Name'] = df.loc[df['Startup Name'].str.contains(search,regex=True),'Startup Name'].replace(r'[\\xe2\\x80\\x99s]','',regex=True)
df.loc[df['Sub Vertical'].isna(),'Sub Vertical'] = 'Unknown'
df.loc[df['Remarks'].isna(),'Remarks'] = 'Unknown'

mask = df[columns].apply(lambda x:x.str.contains(search,regex=True)).any(axis=1)
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\xe2','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\xc2','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\x80','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\x99s','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\x99','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\xa0','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\xc3','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\xa9','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\n','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'\\','',regex=True))
df.loc[mask, columns] = df.loc[mask, columns].apply(lambda col: col.str.replace(r'x98','',regex=True))


## Standardizing city column accordingly 
df['City Location'] = df['City Location'].str.lower()
df.loc[df['City Location'].str.contains('bengaluru'),'City Location'] = 'bangalore'
df.loc[df['City Location'].str.contains('delhi'),'City Location']  = 'new delhi'
df.loc[df['City Location'].str.contains('guru'),'City Location'] = 'gurgaon'
df.loc[df['City Location'].str.contains('indi'),'City Location'] = 'india'
df.loc[df['City Location'].str.contains('bangalore'),'City Location'] = 'bangalore'
df.loc[df['City Location'].str.contains('gurgaon'),'City Location'] = 'gurgaon'
df.loc[df['City Location'].str.contains('mumbai'),'City Location'] = 'mumbai'
df.loc[df['City Location'].str.contains('hyd'),'City Location'] = 'hyderabad'
df.loc[df['City Location'].str.contains('pune'),'City Location'] = 'pune'
df.loc[df['City Location'].str.contains('noida'),'City Location'] = 'noida'
df['City Location'].unique()
df['City Location'] = df['City Location'].str.capitalize()

# Date Conversion and addition of Year to the data for Timeperiod comparision
pd.options.display.float_format = '{:.2f}'.format
from datetime import datetime 
df['Date'] = df['Date'].apply(lambda x : pd.to_datetime(x,dayfirst=True))
df['Year'] = df['Date'].dt.to_period('Y')



In [874]:
## Segeregation of Industry vertical into broader category for easy of understanding 

def classify_services(text):
    text = str(text).lower()
    
    if any(word in text for word in ["e-tech", "technology", "tech", "software", "saas", "ai", "deep-tech", "clean-tech", "iot", "automation", "nanotechnology", "information technology", "consumer technology", "it", "digital media", "aerospace"]):
        return "Technology & Software"
    
    if any(word in text for word in ["e-commerce", "ecommerce", "retail", "consumer goods", "consumer internet", "fmcg", "online marketplace", "auto", "automobile", "automotive"]):
        return 'E-Commerce & Retail'
    
    if any(word in text for word in ["food and beverage", "food & beverage", "food", "food tech", "food-tech", "food & beverages", "food and beverages"]):
        return 'Food & Beverage'
    
    if any(word in text for word in ["fintech", "finance", "fin-tech", "financial tech", "nbfc", "bfsi", "investment", "accounting"]):
        return 'FinTech & Finance'
    
    if any(word in text for word in ["healthcare", "health care", "health and wellness"]):
        return 'Healthcare & Wellness'
    
    if any(word in text for word in ["education", "edtech", "ed-tech", "online education"]):
        return 'EdTech & Education'
    
    if any(word in text for word in ["transportation", "logistics", "transport", "last mile transportation", "logistics tech"]):
        return 'Logistics & Transportation'

    if any(word in text for word in ["real estate"]):
        return 'Real Estate'

    if any(word in text for word in ["video", "gaming", "video games", "social media", "social network", "publishing", "entertainment", "media", "lifestyle"]):
        return 'Media, Social & Entertainment'
    
    if any(word in text for word in ["travel tech", "hospitality"]):
        return 'Travel & Hospitality'
    
    if any(word in text for word in ["fashion and apparel", "fashion", "luxury label"]):
        return 'Fashion & Lifestyle'
    
    if any(word in text for word in ["b2b", "advertising, marketing", "customer service", "customer service platform", "services", "services platform", "compliance", "waste management service", "agriculture", "agtech", "energy"]):
        return 'B2B, Marketing & HR'
# Apply the function
df['Service_Category'] = df['Industry Vertical'].apply(classify_services)

In [875]:
#Which 'Industry Vertical' is attracting the highest 'Average Investment', and how many 'funding gaps' (missing values) exist in that specific sector compared to the rest of the market?"

Industry_vertical = df.groupby('Service_Category').agg(Average_amount = ('Amount in USD','mean'),Count=('Amount in USD','count'),Na_count = ('Amount in USD',lambda x:x.isna().sum())).reset_index()
# Top 10 Industry based on highest avg investment and least funding gap
Industry_vertical = Industry_vertical.sort_values(by =['Average_amount','Count'],ascending =[False,False])
Industry_vertical.head(3)

# The analysis shows higher amount/deal in transportation  followed by FIntech and B2B,Marketing & HR

,Service_Category,Average_amount,Count,Na_count
7,Logistics & Transportation,102879625.40,47,10
4,FinTech & Finance,32763078.12,64,6
0,"B2B, Marketing & HR",30329236.33,45,1


In [876]:
# How has the total funding amount and the number of deals changed year-over-year from 2018 to 2021?
Time_comparision = df.groupby('Year').agg(Investment_Amount =('Amount in USD','sum'),Deals = ('Startup Name','count')).reset_index().sort_values(by=['Investment_Amount','Deals'],ascending=[False,False])
Time_comparision

,Year,Investment_Amount,Deals
2,2017,10429309730.00,687
4,2019,9686576535.22,111
0,2015,8601187368.00,647
3,2018,5122368369.00,310
1,2016,3828088608.00,993
5,2020,390207254.00,7


In [877]:
# Investors want to know the Total Funding vs. the Number of Startups in each city.

City_wise_dist = df.groupby('City Location').agg(Total_funding = ('Amount in USD','sum'),
                                        Total_Startups= ('Startup Name','nunique')).reset_index()
City_wise_dist = City_wise_dist.sort_values(by='Total_funding',ascending = False)
print(f'{City_wise_dist.head(3)}\n')
# Bangalore is becoming the startup hub leading with totalfunding amount and total startups 

# What are the averge and median amount of investment that startup received in India?
Mean = round(df['Amount in USD'].mean(),2)
Median = df['Amount in USD'].median()

print(f'The average amount that a startup received : {Mean}\n')
print(f'The median amount that a startup received :{Median}')

   City Location  Total_funding  Total_Startups
6      Bangalore 18537889363.00             632
43        Mumbai  4932455015.00             451
21       Gurgaon  3872428657.54             250

The average amount that a startup received : 18429897.27

The median amount that a startup received :1700000.0


In [879]:
# Which Service Category attracts the highest number of Unique Investors?
Attractions = df.groupby('Service_Category').agg(Populated =('Amount in USD','count'),
                                                 Avg_Investment=('Amount in USD','mean')).reset_index()
Attractions = Attractions.sort_values(by='Avg_Investment',ascending=False)
Attractions
#Analysis shows the logistic & transportation having highest avg investment per deal though it has lower volume transaction and highest volume of transaction are done in E-commerce & retail  

,Service_Category,Populated,Avg_Investment
7,Logistics & Transportation,47,102879625.40
4,FinTech & Finance,64,32763078.12
0,"B2B, Marketing & HR",45,30329236.33
1,E-Commerce & Retail,830,20417820.41
6,Healthcare & Wellness,60,15834233.33
10,Technology & Software,492,13013566.99
2,EdTech & Education,31,11643000.00
5,Food & Beverage,55,10202658.09
9,Real Estate,10,5080000.00
8,"Media, Social & Entertainment",20,4325818.75


In [915]:
# Leading Investors over the timeperiod 
df[['First Investors','Second Investors','Others']] = df['Investors Name'].str.split(',',n=2,expand=True)
Actual_Investment = df[(df['Amount in USD'].notna())&(df['Investors Name'].str.lower() !='undisclosed investors')] # Investment with Amount invested and not NAN
Investors = Actual_Investment.groupby('First Investors').agg(Deals = ('Startup Name','count'),YTD_Amount=('Amount in USD','sum')).sort_values(by=['Deals','YTD_Amount'],ascending=[False,False])
top_investors = Investors.head(10).index.tolist()
top_investor_data = Actual_Investment[Actual_Investment['Investors Name'].isin(top_investors)]
top_investor_data.groupby(['First Investors','Service_Category']).size()

First Investors          Service_Category     
Accel Partners           E-Commerce & Retail      3
                         Technology & Software    6
Blume Ventures           E-Commerce & Retail      2
                         Healthcare & Wellness    1
                         Technology & Software    1
IDG Ventures             E-Commerce & Retail      1
                         Technology & Software    1
Indian Angel Network     E-Commerce & Retail      6
                         Healthcare & Wellness    1
                         Technology & Software    3
Kalaari Capital          E-Commerce & Retail      4
                         Technology & Software    6
Nexus Venture Partners   E-Commerce & Retail      3
                         Technology & Software    2
SAIF Partners            E-Commerce & Retail      2
                         Food & Beverage          1
                         Technology & Software    1
Sequoia Capital          E-Commerce & Retail      4
                 